# 📋 Documentación Técnica: Simulador de Cancelación Total
## Comparativa: Código Original vs Código Refactorizado
---

## 1. Contexto General

Sistema de simulación de cancelación total de créditos en plataforma bancaria. Permite negociar descuentos sobre intereses y capital según edad de mora y mecanismos de campaña.

**Problema identificado:** El código original tenía múltiples bugs, inconsistencias en sessionStorage y lógica de distribución proporcional incorrecta.

**Solución:** Refactorización completa con:
- Lógica del "Rombo" para distribución proporcional
- Bug fixes documentados
- Código modular y mantenible

---

## 2. GUIDs de Campos del Formulario

| Campo | GUID | Descripción |
|-------|------|-------------|
| Saldo Total | `f47f1a89-6743-4f60-9cf6-0696e6c841ca` | Total del crédito |
| Int corrientes | `48f8260e-5e81-43d3-b69c-d94808cb229e` | Interés corriente |
| Max Baja Int Cte | `1c23cf01-dd67-4c9a-a4b6-871c781eec02` | Tope descuento Int Cte |
| Baja Int Cte | `86f86bd7-d119-4d2a-a6c0-e711b1d835a6` | Descuento aplicado Int Cte |
| %Baja Int Cte | `bcfd54b6-d1cf-40dc-8677-686652eedbb8` | Porcentaje descuento Int Cte |
| Interés en Mora | `d85c85c3-2a7c-44db-b240-2420990d7375` | Interés por mora |
| Max Baja Int Mora | `0c422603-5f6e-4c23-a7b5-b78cf30ba1d8` | Tope descuento Int Mora |
| Baja Int Mora | `a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115` | Descuento aplicado Int Mora |
| % Baja Int Mora | `433ffa22-78e7-4004-be47-2b0ccf497ad1` | Porcentaje descuento Int Mora |
| Int Extra TC | `a9977387-4683-4d89-9e58-851cb72f9886` | Int extracontable TC |
| Max Baja Int Extra TC | `6e01ec4d-1391-4878-8886-be49eef96d27` | Tope descuento Int Extra |
| Baja Int Extra TC | `8ea64929-53a9-41b4-a01f-a14b74293d01` | Descuento aplicado Int Extra |
| % Dcsc Int Extra TC | `a724067d-e7bf-435c-94ac-bf44f72575e7` | Porcentaje descuento Int Extra |
| Capital Total | `9dc154b0-5d64-4682-a76d-5e946415c253` | Monto de capital |
| Max Baja Capital | `79b6141f-1973-4c00-b0ae-a26b657115e5` | Tope descuento capital |
| Baja Capital | `60bebeab-d3ca-4547-9eff-00cc8db69b82` | Descuento aplicado capital |
| % Baja Capital | `aa7aeaf3-6bc8-4939-9896-212d5efcd93e` | Porcentaje descuento capital |
| Max Baja en cuentas | `7a94fe37-1d84-4232-9298-4e1986cdead2` | Suma de máximos descuentos |
| Abono mínimo | `0864b793-256f-41f6-ab7c-5b5c18c1f51f` | Pago mínimo para max % |
| Bajas aplicadas | `7ed52d26-15c9-4f11-9177-55a380d1427d` | Total descuentos aplicados |
| Pago al SNR | `b5c33a6d-9d65-4920-8a39-e73621b7daa9` | Pago realizado por cliente |
| Es Tarjeta | `876c30bc-ba27-4ec4-ad2e-1635b23cdccb` | Flag producto TARJETA |

---

## 3. Variables en sessionStorage

| Variable | Descripción |
|----------|-------------|
| `edadmoraCancelacion` | Edad de mora del crédito |
| `PorcCancelacionIntCte` | % máximo descuento Int Cte |
| `PorcCancelacionIntMora` | % máximo descuento Int Mora |
| `pordescuentoextra` | % máximo descuento Int Extra TC |
| `PorcentajeCT` | % máximo descuento Capital |
| `campañacancelacion` | Flag 1 si hay campaña activa |
| `MecanismoAplicaCampana` | Código de campaña |
| `DctoCapitalCampana` | % descuento capital campaña |
| `desCapital` | Valor $ descuento inicial capital |
| `Capital` | Flag "Si" si hay dcto capital |

---

## 4. Bugs Corregidos

### Bug #1: `pordescuentoextra` copiaba valor de mora

**ANTES (línea 118 OLD):**
```javascript
let pordescuentoextra = sessionStorage.PorcCancelacionIntMora;  // ❌ Copia mora
```

**DESPUÉS:**
```javascript
let pordescuentoextra = tasas.PorcPagoMoraIntExtraC;  // ✓ Variable correcta
// Y luego:
sessionStorage.pordescuentoextra = pordescuentoextra;
```

### Bug #2: Descuento Int Extra usaba tasa de Mora

**ANTES (línea 266 OLD):**
```javascript
let maxdescuentointeresExtraTC = (sessionStorage.PorcCancelacionIntMora / 100) * interesExtraTC
// ❌ Usa PorcCancelacionIntMora en lugar de PorcPagoMoraIntExtraC
```

**DESPUÉS:**
```javascript
let maxDctoExtra = bloqueado ? 0 : (porcExtra / 100) * interesextra
// ✓ Usa porcExtra correctamente desde sessionStorage.pordescuentoextra
```

### Bug #3: Inconsistencia en nombre de edad de mora

**ANTES:**
```javascript
let edad = sessionStorage.EdadMoraCl  // En algunos lugares
let edad = sessionStorage.edadmoraCancelacion  // En otros
```

**DESPUÉS:**
```javascript
let edad = sessionStorage.edadmoraCancelacion  // ✓ Unificado en toda la cadena
```

### Bug #4: Falta validación de query

**ANTES:**
```javascript
// No había validación, podía romper si no había datos
let tasas = response[0][0]
```

**DESPUÉS:**
```javascript
if (!response || !response[0] || !response[0][0]) {
    console.error('No se encontraron tasas para edad de mora...')
    toastr.warning('No se encontraron tasas...')
    return
}
let tasas = response[0][0]
```

### Bug #5: Lógica de descuento proporcional incorrecta

**ANTES:**
- No había cálculo de presupuesto proporcional
- Los descuentos se mantenían al máximo sin importar el pago
- Cálculos dispersos en múltiples funciones

**DESPUÉS:**
- Nueva **"Lógica del Rombo"** que calcula distribución proporcional
- Orden: Mora → (Cte + Extra) → Capital
- Porcentajes se ajustan automáticamente

---

## 5. Código ANTES (OLD) - `poblarCancelacion()`

```javascript
function vaciacancelar() { }
async function poblarCancelacion() {
    delete sessionStorage.Capital
    let producto = e.dataItem.Producto
    let saldoTotalObl = e.dataItem.SaldoTotalObl
    setFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca', saldoTotalObl)
    let interescteObl = e.dataItem.InteresCteObl
    setFieldValue('48f8260e-5e81-43d3-b69c-d94808cb229e', interescteObl)
    let edadmora = e.dataItem.EdadMoraCl
    sessionStorage.edadmoraCancelacion = edadmora
    // Consulta tasas vigentes por edad de mora
    let query = `SELECT PorcCancelacionIntMora, PorcCancelacionIntCte, PorcPagoMoraIntExtraC, PorcentajeCT from SimiladorDNC_Lappiz_TasasVertas where RangoDias3 = '${edadmora}'`
    let response = await execQuery(query)
    console.log(response[0][0])
    sessionStorage.PorcCancelacionIntCte = response[0][0].PorcCancelacionIntCte
    sessionStorage.PorcCancelacionIntMora = response[0][0].PorcCancelacionIntMora
    sessionStorage.PorcPagoMoraIntExtraC = response[0][0].PorcPagoMoraIntExtraC
    sessionStorage.PorcentajeCT = response[0][0].PorcentajeCT
    let Pordescuentoint = response[0][0].PorcCancelacionIntCte
    let pordescuentomora = response[0][0].PorcCancelacionIntMora
    let pordescuentoextra = response[0][0].PorcPagoMoraIntExtraC
    let PorcentajeCT = response[0][0].PorcentajeCT
    let desCapital = 0
    sessionStorage.MecanismoAplicaCampana = e.dataItem.MecanismoAplicaCampana
    if (e.dataItem.MecanismoAplicaCampana && e.dataItem.MecanismoAplicaCampana.includes("CANCELACION")) {
        sessionStorage.campañacancelacion = 1
        Pordescuentoint = e.dataItem.DtoInteresesCampana
        pordescuentomora = e.dataItem.DtoInteresesMoraCampana
        pordescuentoextra = e.dataItem.DtoInteresExtracontablesCampana
        PorcentajeCT = e.dataItem.DctoCapitalCampana
        console.log('si tiene campaña en cancelacion')
        desCapital = e.dataItem.CapitalTotalObl * (e.dataItem.DctoCapitalCampana / 100)
        sessionStorage.DctoCapitalCampana = e.dataItem.DctoCapitalCampana / 100
        sessionStorage.Capital = 'Si'
    }
    sessionStorage.PorcCancelacionIntCte = Pordescuentoint
    sessionStorage.PorcCancelacionIntMora = pordescuentomora
    sessionStorage.desCapital = desCapital
    sessionStorage.PorcentajeCT = PorcentajeCT
    sessionStorage.Pordescuentoint = Pordescuentoint
    sessionStorage.pordescuentomora = pordescuentomora
    sessionStorage.pordescuentoextra = pordescuentoextra
    setFieldValue('bcfd54b6-d1cf-40dc-8677-686652eedbb8', Pordescuentoint)
    // Porcentaje descuento interés corriente (siempre se llena)
    let maxDescuentoInt = (Pordescuentoint / 100) * interescteObl
    setFieldValue('1c23cf01-dd67-4c9a-a4b6-871c781eec02', maxDescuentoInt)
    // Interés mora (siempre se llena)
    let interesmora = e.dataItem.InteresMoraObl
    setFieldValue('d85c85c3-2a7c-44db-b240-2420990d7375', interesmora)
    let maxdescuentomora = (pordescuentomora / 100) * interesmora
    setFieldValue('0c422603-5f6e-4c23-a7b5-b78cf30ba1d8', maxdescuentomora)
    setFieldValue('a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115', maxdescuentomora)
    setFieldValue('433ffa22-78e7-4004-be47-2b0ccf497ad1', pordescuentomora)
    // Interés extracontable (solo para TARJETA)
    let interesExtraTC = 0
    if (producto == 'TARJETA') {
        interesExtraTC = e.dataItem.InteresesExtracontablesObl
    }
    setFieldValue('a9977387-4683-4d89-9e58-851cb72f9886', interesExtraTC)
    let maxdescuentointeresExtraTC = (pordescuentoextra / 100) * interesExtraTC
    setFieldValue('6e01ec4d-1391-4878-8886-be49eef96d27', maxdescuentointeresExtraTC)
    setFieldValue('8ea64929-53a9-41b4-a01f-a14b74293d01', maxdescuentointeresExtraTC)
    setFieldValue('a724067d-e7bf-435c-94ac-bf44f72575e7', pordescuentoextra)
    // Capital (siempre se llena)
    let capital = e.dataItem.CapitalTotalObl
    setFieldValue('9dc154b0-5d64-4682-a76d-5e946415c253', capital)
    let maxdescuentocapital = (PorcentajeCT / 100) * capital
    setFieldValue('79b6141f-1973-4c00-b0ae-a26b657115e5', maxdescuentocapital)
    setFieldValue('60bebeab-d3ca-4547-9eff-00cc8db69b82', desCapital)
    let porcentajedescuentocapital = desCapital / capital
    setFieldValue('aa7aeaf3-6bc8-4939-9896-212d5efcd93e', porcentajedescuentocapital)
    // Total descuentos (siempre se calcula)
    let maxdescuentos = maxdescuentocapital + maxdescuentointeresExtraTC + maxdescuentomora + maxDescuentoInt
    setFieldValue('7a94fe37-1d84-4232-9298-4e1986cdead2', maxdescuentos)
    // Abono mínimo (siempre se calcula)
    let abonominimo = e.dataItem.SaldoTotalObl - maxdescuentos + 10000
    setFieldValue('0864b793-256f-41f6-ab7c-5b5c18c1f51f', abonominimo)
    // EsTarjeta
    if (producto == 'TARJETA') {
        setFieldValue('876c30bc-ba27-4ec4-ad2e-1635b23cdccb', 'Si')
    } else {
        setFieldValue('876c30bc-ba27-4ec4-ad2e-1635b23cdccb', 'No')
    }
}
```

**Problemas identificados:**
- No había validación de query
- No se bloqueaban descuentos para 1-30 y 31-60 días
- Inconsistencia en nombres de sessionStorage

---

## 6. Código DESPUÉS (Refactorizado) - `poblarCancelacion()`

```javascript
// ─────────────────────────────────────────────────────────────
// POBLAR CANCELACIÓN — carga inicial desde la fila seleccionada
// ─────────────────────────────────────────────────────────────
function vaciacancelar() { }
async function poblarCancelacion() {
    delete sessionStorage.Capital
    let producto = e.dataItem.Producto
    let saldoTotalObl = e.dataItem.SaldoTotalObl
    let interescteObl = e.dataItem.InteresCteObl
    let interesmora = e.dataItem.InteresMoraObl
    let capital = e.dataItem.CapitalTotalObl
    let edadmora = e.dataItem.EdadMoraCl
    sessionStorage.edadmoraCancelacion = edadmora
    // ── Consulta tasas ──
    let query = `SELECT PorcCancelacionIntMora, PorcCancelacionIntCte, PorcPagoMoraIntExtraC, PorcentajeCT 
                 FROM SimiladorDNC_Lappiz_TasasVertas WHERE RangoDias3 = '${edadmora}'`
    let response = await execQuery(query)
    //  Validar que la query devolvió datos
    if (!response || !response[0] || !response[0][0]) {
        console.error(`poblarCancelacion: No se encontraron tasas para edad de mora "${edadmora}". 
                       Verificar tabla SimiladorDNC_Lappiz_TasasVertas y columna RangoDias3.`)
        toastr.warning(`No se encontraron tasas para la edad de mora: "${edadmora}"`)
        return
    }
    let tasas = response[0][0]
    let Pordescuentoint = tasas.PorcCancelacionIntCte
    let pordescuentomora = tasas.PorcCancelacionIntMora
    let pordescuentoextra = tasas.PorcPagoMoraIntExtraC
    let PorcentajeCT = tasas.PorcentajeCT
    let desCapital = 0
    sessionStorage.MecanismoAplicaCampana = e.dataItem.MecanismoAplicaCampana
    if (e.dataItem.MecanismoAplicaCampana && e.dataItem.MecanismoAplicaCampana.includes("CANCELACION")) {
        sessionStorage.campañacancelacion = 1
        Pordescuentoint = e.dataItem.DtoInteresesCampana
        pordescuentomora = e.dataItem.DtoInteresesMoraCampana
        pordescuentoextra = e.dataItem.DtoInteresExtracontablesCampana
        PorcentajeCT = e.dataItem.DctoCapitalCampana
        desCapital = capital * (e.dataItem.DctoCapitalCampana / 100)
        sessionStorage.DctoCapitalCampana = e.dataItem.DctoCapitalCampana / 100
        sessionStorage.Capital = 'Si'
    }
    // Guardar en session
    sessionStorage.PorcCancelacionIntCte = Pordescuentoint
    sessionStorage.PorcCancelacionIntMora = pordescuentomora
    sessionStorage.pordescuentoextra = pordescuentoextra
    sessionStorage.PorcentajeCT = PorcentajeCT
    sessionStorage.Pordescuentoint = Pordescuentoint
    sessionStorage.pordescuentomora = pordescuentomora
    sessionStorage.desCapital = desCapital
    let bloqueado = (edadmora === "1-30 Días" || edadmora === "31-60 Días")
    // ── Saldo total ──
    setFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca', saldoTotalObl)
    // ── Int corriente ──
    setFieldValue('48f8260e-5e81-43d3-b69c-d94808cb229e', interescteObl)
    let maxDctoIntCte = bloqueado ? 0 : (Pordescuentoint / 100) * interescteObl
    setFieldValue('1c23cf01-dd67-4c9a-a4b6-871c781eec02', maxDctoIntCte)
    setFieldValue('86f86bd7-d119-4d2a-a6c0-e711b1d835a6', maxDctoIntCte)
    setFieldValue('bcfd54b6-d1cf-40dc-8677-686652eedbb8', bloqueado ? 0 : Pordescuentoint)
    // ── Int mora ──
    setFieldValue('d85c85c3-2a7c-44db-b240-2420990d7375', interesmora)
    let maxDctoMora = bloqueado ? 0 : (pordescuentomora / 100) * interesmora
    setFieldValue('0c422603-5f6e-4c23-a7b5-b78cf30ba1d8', maxDctoMora)
    setFieldValue('a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115', maxDctoMora)
    setFieldValue('433ffa22-78e7-4004-be47-2b0ccf497ad1', bloqueado ? 0 : pordescuentomora)
    // ── Int extracontable TC ──
    let interesExtraTC = (producto === 'TARJETA') ? e.dataItem.InteresesExtracontablesObl : 0
    setFieldValue('a9977387-4683-4d89-9e58-851cb72f9886', interesExtraTC)
    let maxDctoExtra = bloqueado ? 0 : (pordescuentoextra / 100) * interesExtraTC
    setFieldValue('6e01ec4d-1391-4878-8886-be49eef96d27', maxDctoExtra)
    setFieldValue('8ea64929-53a9-41b4-a01f-a14b74293d01', maxDctoExtra)
    setFieldValue('a724067d-e7bf-435c-94ac-bf44f72575e7', bloqueado ? 0 : pordescuentoextra)
    // ── Capital ──
    setFieldValue('9dc154b0-5d64-4682-a76d-5e946415c253', capital)
    let maxDctoCapital = bloqueado ? 0 : (PorcentajeCT / 100) * capital
    setFieldValue('79b6141f-1973-4c00-b0ae-a26b657115e5', maxDctoCapital)
    setFieldValue('60bebeab-d3ca-4547-9eff-00cc8db69b82', bloqueado ? 0 : desCapital)
    let porCapitalInicial = (capital > 0 && !bloqueado) ? (desCapital / capital) * 100 : 0
    setFieldValue('aa7aeaf3-6bc8-4939-9896-212d5efcd93e', porCapitalInicial)
    // ── Totales ──
    let totalMaxDctos = maxDctoIntCte + maxDctoExtra + maxDctoMora + maxDctoCapital
    setFieldValue('7a94fe37-1d84-4232-9298-4e1986cdead2', totalMaxDctos)
    let abonoMinimo = saldoTotalObl - totalMaxDctos + 10000
    setFieldValue('0864b793-256f-41f6-ab7c-5b5c18c1f51f', abonoMinimo)
    setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', totalMaxDctos)
    // ── Es Tarjeta ──
    setFieldValue('876c30bc-ba27-4ec4-ad2e-1635b23cdccb', producto === 'TARJETA' ? 'Si' : 'No')
    console.log('poblarCancelacion OK ✓', { edadmora, tasas, bloqueado })
}
```

**Mejoras:**
- ✅ Validación de query con mensaje de error
- ✅ Variable `bloqueado` unificada para edades 1-30 y 31-60 días
- ✅ Todos los descuentos se bloquean correctamente
- ✅ Uso correcto de `pordescuentoextra`
- ✅ Porcentaje capital calculado como % (no decimal)

---

## 7. Código OLD - `recalcularcancelacion()`

```javascript
function recalcularcancelacion() {
    debugger
    let interescteObl = getFieldValue('48f8260e-5e81-43d3-b69c-d94808cb229e')
    let maxDescuentoInt = (sessionStorage.PorcCancelacionIntCte / 100) * interescteObl
    let edad = sessionStorage.EdadMoraCl  // ❌ Debería ser edadmoraCancelacion
    if (edad == "1-30 Días" || edad == "31-60 Días") {
        maxDescuentoInt = 0
    }
    setFieldValue('1c23cf01-dd67-4c9a-a4b6-871c781eec02', maxDescuentoInt)
    let Pordescuentoint = parseFloat(sessionStorage.getItem('Pordescuentoint'));
    let pordescuentomora = parseFloat(sessionStorage.getItem('pordescuentomora'));
    let pordescuentoextra = parseFloat(sessionStorage.getItem('pordescuentoextra'));  // ❌ Nunca se guardó así
    let desCapital = parseFloat(sessionStorage.getItem('desCapital'));
    setFieldValue('bcfd54b6-d1cf-40dc-8677-686652eedbb8', Pordescuentoint)
    //seccion mora
    let interesmora = getFieldValue('d85c85c3-2a7c-44db-b240-2420990d7375')
    let maxdescuentomora = (sessionStorage.PorcCancelacionIntMora / 100) * interesmora
    if (edad == "1-30 Días" || edad == "31-60 Días") {
        maxdescuentomora = 0;
    }
    setFieldValue('0c422603-5f6e-4c23-a7b5-b78cf30ba1d8', maxdescuentomora)
    setFieldValue('a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115', maxdescuentomora)
    setFieldValue('433ffa22-78e7-4004-be47-2b0ccf497ad1', pordescuentomora)
    //seccion extracontables
    let interesExtraTC = getFieldValue('a9977387-4683-4d89-9e58-851cb72f9886')
    // ❌ BUG: Usa PorcCancelacionIntMora en lugar de PorcPagoMoraIntExtraC
    let maxdescuentointeresExtraTC = (sessionStorage.PorcCancelacionIntMora / 100) * interesExtraTC
    if (edad == "1-30 Días" || edad == "31-60 Días") {
        maxdescuentointeresExtraTC = 0;
    }
    setFieldValue('6e01ec4d-1391-4878-8886-be49eef96d27', maxdescuentointeresExtraTC)
    setFieldValue('8ea64929-53a9-41b4-a01f-a14b74293d01', maxdescuentointeresExtraTC)
    setFieldValue('a724067d-e7bf-435c-94ac-bf44f72575e7', pordescuentoextra)
    let saldoT = getFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca')
    let snr = getFieldValue('b5c33a6d-9d65-4920-8a39-e73621b7daa9')
    let capital = getFieldValue('9dc154b0-5d64-4682-a76d-5e946415c253')
    let maxdescuentocapital = (sessionStorage.PorcentajeCT / 100) * capital
    if (edad == "1-30 Días" || edad == "31-60 Días") {
        maxdescuentocapital = 0;
        desCapital = 0;
    }
    setFieldValue('79b6141f-1973-4c00-b0ae-a26b657115e5', maxdescuentocapital)
    setFieldValue('60bebeab-d3ca-4547-9eff-00cc8db69b82', desCapital)
    let porcentajedescuentocapital = (desCapital / capital)
    porcentajedescuentocapital = Math.round((porcentajedescuentocapital * 100))
    setFieldValue('aa7aeaf3-6bc8-4939-9896-212d5efcd93e', porcentajedescuentocapital)
    let saldoTotalObl = getFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca')
    let maxdescuentos = maxdescuentocapital + maxdescuentointeresExtraTC + maxdescuentomora + maxDescuentoInt
    setFieldValue('7a94fe37-1d84-4232-9298-4e1986cdead2', maxdescuentos)
    let operacion = saldoT - snr + 10000
    if (operacion > 0) {
        if (edad == "1-30 Días" || edad == "31-60 Días") {
            operacion = 0;
        }
        setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', operacion)
    } else {
        setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', 0)
    }
    let descuentooperacion = DescuentoInteresCte()
    setFieldValue('86f86bd7-d119-4d2a-a6c0-e711b1d835a6', descuentooperacion)
}
```

**Problemas:**
- ❌ Usa `sessionStorage.EdadMoraCl` en lugar de `sessionStorage.edadmoraCancelacion`
- ❌ BUG: Usa `PorcCancelacionIntMora` para calcular descuento de Int Extra TC
- ❌ Lógica de distribución proporcional incorrecta
- ❌ Depende de función `DescuentoInteresCte()` que también tenía bugs

---

## 8. Código DESPUÉS - `recalcularcancelacion()` con Lógica del Rombo

```javascript
// ─────────────────────────────────────────────────────────────
// RECALCULAR CANCELACIÓN
// Distribuye el presupuesto de descuentos según el pago al SNR
// Orden: Int Cte → Int Extra TC → Int Mora → Capital
// ─────────────────────────────────────────────────────────────
function recalcularcancelacion() {
    let edad = sessionStorage.edadmoraCancelacion
    let bloqueado = (edad === "1-30 Días" || edad === "31-60 Días")
    // ── Leer valores del formulario ──
    let saldoTotal = getFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca')
    let pagoAlSnr = getFieldValue('b5c33a6d-9d65-4920-8a39-e73621b7daa9')
    let capital = getFieldValue('9dc154b0-5d64-4682-a76d-5e946415c253')
    let interescte = getFieldValue('48f8260e-5e81-43d3-b69c-d94808cb229e')
    let interesmora = getFieldValue('d85c85c3-2a7c-44db-b240-2420990d7375')
    let interesextra = getFieldValue('a9977387-4683-4d89-9e58-851cb72f9886')
    // ── Porcentajes máximos desde session ──
    let porcCte = parseFloat(sessionStorage.PorcCancelacionIntCte) || 0
    let porcMora = parseFloat(sessionStorage.PorcCancelacionIntMora) || 0
    let porcExtra = parseFloat(sessionStorage.pordescuentoextra) || 0
    let porcCapital = parseFloat(sessionStorage.PorcentajeCT) || 0
    // ── Descuentos máximos permitidos (topes fijos) ──
    let maxDctoIntCte = bloqueado ? 0 : (porcCte / 100) * interescte
    let maxDctoMora = bloqueado ? 0 : (porcMora / 100) * interesmora
    let maxDctoExtra = bloqueado ? 0 : (porcExtra / 100) * interesextra
    let maxDctoCapital = bloqueado ? 0 : (porcCapital / 100) * capital
    // Setear campos Max Baja (topes, siempre fijos)
    setFieldValue('1c23cf01-dd67-4c9a-a4b6-871c781eec02', maxDctoIntCte)
    setFieldValue('0c422603-5f6e-4c23-a7b5-b78cf30ba1d8', maxDctoMora)
    setFieldValue('6e01ec4d-1391-4878-8886-be49eef96d27', maxDctoExtra)
    setFieldValue('79b6141f-1973-4c00-b0ae-a26b657115e5', maxDctoCapital)
    // ── Abono mínimo (pago que cubre el 100% de descuentos) ──
    let totalMaxDctos = maxDctoIntCte + maxDctoExtra + maxDctoMora + maxDctoCapital
    let abonoMinimo = saldoTotal - totalMaxDctos + 10000
    setFieldValue('7a94fe37-1d84-4232-9298-4e1986cdead2', totalMaxDctos)
    setFieldValue('0864b793-256f-41f6-ab7c-5b5c18c1f51f', abonoMinimo)
    // ── Si está bloqueado, todo en 0 y salir ──
    if (bloqueado) {
        setFieldValue('86f86bd7-d119-4d2a-a6c0-e711b1d835a6', 0)
        setFieldValue('bcfd54b6-d1cf-40dc-8677-686652eedbb8', 0)
        setFieldValue('a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115', 0)
        setFieldValue('433ffa22-78e7-4004-be47-2b0ccf497ad1', 0)
        setFieldValue('8ea64929-53a9-41b4-a01f-a14b74293d01', 0)
        setFieldValue('a724067d-e7bf-435c-94ac-bf44f72575e7', 0)
        setFieldValue('60bebeab-d3ca-4547-9eff-00cc8db69b82', 0)
        setFieldValue('aa7aeaf3-6bc8-4939-9896-212d5efcd93e', 0)
        setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', 0)
        return
    }
    // ════════════════════════════════════════════════════════
    // LÓGICA DEL ROMBO
    // 
    // Abono mínimo → descuentos al 100% (rombo vacío)
    // Cliente paga MÁS → rombo se llena → descuentos bajan
    // Orden de llenado: Mora → (Cte + Extra juntos) → Capital
    //
    // "pagoExtra" = cuánto pagó el cliente POR ENCIMA del abono mínimo
    // ════════════════════════════════════════════════════════
    let pagoExtra = Math.max(0, pagoAlSnr - abonoMinimo)
    // Variables de descuento aplicado (inician en máximo y bajan)
    let dctoCte = maxDctoIntCte
    let dctoMora = maxDctoMora
    let dctoExtra = maxDctoExtra
    let dctoCapital = maxDctoCapital
    let exceso = pagoExtra
    // ── Paso 1: Llenar mora primero ──
    if (exceso > 0) {
        let reduccionMora = Math.min(maxDctoMora, exceso)
        dctoMora = maxDctoMora - reduccionMora
        exceso -= reduccionMora
    }
    // ── Paso 2: Llenar Int Cte e Int Extra juntos (misma proporción) ──
    if (exceso > 0) {
        let maxCteExtra = maxDctoIntCte + maxDctoExtra
        if (maxCteExtra > 0) {
            let reduccionCteExtra = Math.min(maxCteExtra, exceso)
            let porcionRestante = (maxCteExtra - reduccionCteExtra) / maxCteExtra
            dctoCte = maxDctoIntCte * porcionRestante
            dctoExtra = maxDctoExtra * porcionRestante
            exceso -= reduccionCteExtra
        }
    }
    // ── Paso 3: Llenar capital (tope al máximo permitido) ──
    if (exceso > 0) {
        let reduccionCapital = Math.min(maxDctoCapital, exceso)
        dctoCapital = maxDctoCapital - reduccionCapital
        dctoCapital = Math.max(0, dctoCapital)
    }
    // ── Calcular porcentajes reales aplicados ──
    let porCteReal = interescte > 0 ? (dctoCte / interescte) * 100 : 100
    let porMoraReal = interesmora > 0 ? (dctoMora / interesmora) * 100 : 100
    let porExtraReal = interesextra > 0 ? (dctoExtra / interesextra) * 100 : 100
    let porCapitalReal = capital > 0 ? (dctoCapital / capital) * 100 : 0
    // Verificar que Cte y Extra tengan el mismo %
    if (interescte > 0 && interesextra > 0) {
        porExtraReal = porCteReal
        dctoExtra = (porCteReal / 100) * interesextra
    }
    // ── Setear descuentos aplicados ──
    setFieldValue('86f86bd7-d119-4d2a-a6c0-e711b1d835a6', dctoCte)
    setFieldValue('bcfd54b6-d1cf-40dc-8677-686652eedbb8', porCteReal)
    setFieldValue('a6ee4c8b-a6c5-4bd8-8c30-9e29b9c40115', dctoMora)
    setFieldValue('433ffa22-78e7-4004-be47-2b0ccf497ad1', porMoraReal)
    setFieldValue('8ea64929-53a9-41b4-a01f-a14b74293d01', dctoExtra)
    setFieldValue('a724067d-e7bf-435c-94ac-bf44f72575e7', porExtraReal)
    setFieldValue('60bebeab-d3ca-4547-9eff-00cc8db69b82', dctoCapital)
    setFieldValue('aa7aeaf3-6bc8-4939-9896-212d5efcd93e', porCapitalReal)
    // ── Bajas en cuentas aplicadas ──
    let totalAplicado = dctoCte + dctoExtra + dctoMora + dctoCapital
    setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', totalAplicado)
}
```

**Mejoras:**
- ✅ Uso correcto de `sessionStorage.edadmoraCancelacion`
- ✅ Uso correcto de `pordescuentoextra` para Int Extra TC
- ✅ Lógica del Rombo implementada correctamente
- ✅ Porcentajes se calculan proporcionalmente
- ✅ Código modular y fácil de mantener

---

## 9. Flujo Funcional - Value Change del Pago al SNR

### ANTES (OLD):
```javascript
if (e.value > 0) {
    debugger; DescuentoInteresCte()
    recalcularcancelacion()
    if (e.value < getFieldValue('0864b793-256f-41f6-ab7c-5b5c18c1f51f')) {
        toastr.warning('El valor a pagar debe ser mayor al abono minimo que debe realizar el cliente.')
    }
    let descuento = getFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca')
    let operacion = descuento + 10000 - e.value
    if (operacion > 0) {
        setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', operacion)
    } else {
        setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', 0)
    }
}
```

### DESPUÉS:
```javascript
if (e.value <= 0) {
    // No hacer nada
} else {
    recalcularcancelacion();
    let abonoMinimo = getFieldValue('0864b793-256f-41f6-ab7c-5b5c18c1f51f')
    if (e.value < abonoMinimo) {
        toastr.warning('El valor a pagar debe ser mayor al abono mínimo que debe realizar el cliente.')
    }
    let saldoTotal = getFieldValue('f47f1a89-6743-4f60-9cf6-0696e6c841ca')
    let operacion = saldoTotal + 10000 - e.value
    setFieldValue('7ed52d26-15c9-4f11-9177-55a380d1427d', operacion > 0 ? operacion : 0)
}
```

---

## 10. Casos de Uso - Lógica del Rombo

### Caso 1: TC sin intereses, pago al máximo (abono mínimo)

**Entrada:**
- Saldo Total: $1,761,459
- Capital: $1,601,372
- Int Cte: $0
- Int Mora: $2,850
- Int Extra TC: $5,236
- Porcentajes: 100%, 100%, 100%, 5%
- **Pago al SNR: $1,683,305** (exacto al abono mínimo)

**Cálculo:**
```
Presupuesto = $1,761,459 + $10,000 − $1,683,305 = $88,154

Max descuentos:
  maxDctoIntCte    = 0% × $0 = $0
  maxDctoMora      = 100% × $2,850 = $2,850
  maxDctoExtra     = 100% × $5,236 = $5,236
  maxDctoCapital   = 5% × $1,601,372 = $80,068
  Total max       = $0 + $2,850 + $5,236 + $80,068 = $88,154

pagoExtra = $1,683,305 - $1,683,305 = $0

Paso 1: Mora = $2,850 - $0 = $2,850
Paso 2: Cte+Extra = $0 + $5,236 = $5,236 (sin reducción)
Paso 3: Capital = $80,068 - $0 = $80,068
```

**Resultado:** Todos los descuentos al 100%

### Caso 2: TC, pago PARCIAL (paga más del mínimo)

**Entrada (misma que Caso 1, pero):**
- **Pago al SNR: $1,700,000** (mayor al abono mínimo)

**Cálculo:**
```
abonoMinimo = $1,683,305
pagoExtra = $1,700,000 - $1,683,305 = $16,695

Paso 1: Mora = $2,850 - $2,850 = $0 (se llena primero)
         exceso = $16,695 - $2,850 = $13,845

Paso 2: Cte+Extra = $5,236 - $13,845 → No se puede, máximo es $5,236
         Cte = $0, Extra = $0 (ambos se llenan)
         exceso = $13,845 - $5,236 = $8,609

Paso 3: Capital = $80,068 - $8,609 = $71,459
```

**Resultado:**
- Mora: 0% (se perdió por pagar de más)
- Extra: 0% (se perdió por pagar de más)
- Capital: 4.46% (reducido)

### Caso 3: Pago menor al mínimo (warning)

**Entrada:**
- **Pago al SNR: $1,500,000** (menor al abono mínimo $1,683,305)

**Cálculo:**
```
pagoExtra = $1,500,000 - $1,683,305 = 0 (no hay exceso)

Como no hay exceso, los descuentos se mantienen al máximo:
- Mora: 100%
- Extra: 100%
- Capital: 5%

Se muestra warning: 'El valor a pagar debe ser mayor al abono mínimo...'
```

**Resultado:** Descuentos al máximo, pero se muestra warning al usuario

### Caso 4: Crédito bloqueado (1-30 días)

**Entrada:**
- Edad de mora: "1-30 Días"

**Cálculo:**
```
bloqueado = true

Todos los descuentos se ponen a 0:
- dctoCte = 0
- dctoMora = 0
- dctoExtra = 0
- dctoCapital = 0
- porCteReal = 0
- porMoraReal = 0
- porExtraReal = 0
- porCapitalReal = 0
- totalAplicado = 0

Se retorna inmediatamente, sin lógica del rombo
```

**Resultado:** No hay descuentos disponibles para esta edad de mora

---

## 11. Resumen de Cambios Principales

| Aspecto | ANTES (OLD) | DESPUÉS |
|---------|-------------|---------|
| Edad mora | `sessionStorage.EdadMoraCl` inconsistente | `sessionStorage.edadmoraCancelacion` unificado |
| Bug Int Extra | Usaba `PorcCancelacionIntMora` | Usa `pordescuentoextra` correcto |
| Validación query | Sin validación | Con validación y mensaje |
| Bloqueo 1-30/31-60 | Manual en cada sección | Variable `bloqueado` unificada |
| Lógica proporcional | Incorrecta/dispersa | Lógica del Rombo implementada |
| Funciones auxiliares | `DescuentoInteresCte()` con bugs | Eliminadas, lógica centralizada |
| Porcentaje capital | Decimal (0.05) | Porcentaje (5) |
| Código | 344 líneas | 299 líneas |
| Consola | `debugger` olvidado | Logs estructurados con ✓ |

---

## 12. Checklist de Verificación

- [x] Código refactorizado y documentado
- [ ] Verificar que `poblarCancelacion()` se llama al seleccionar crédito
- [ ] Verificar que `recalcularcancelacion()` está asignado al value change del Pago al SNR
- [ ] Probar casos: pago máximo, pago parcial, pago mínimo
- [ ] Validar que bloqueo (edad 1-60 días) funciona correctamente
- [ ] Probar con campañas CANCELACION activas
- [ ] Validar números en calculadora interactiva
- [ ] ¿Hay valuechange en Capital Total? Si existe, pasarlo para revisión